# Gradient Checkpointing

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Activations dominate memory in deep nets. Gradient checkpointing drops activations during the forward pass and recomputes them during backward, trading roughly 33 % more compute for a huge memory reduction. Lets you train models that wouldn't otherwise fit.


## Mathematical Formulation

Memory: $O(N)$ → $O(\sqrt{N})$ when checkpointing every $\sqrt{N}$ layers (segmented checkpointing).


## Implementation


In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint


In [ ]:
class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, x): return self.net(x) + x

class Stack(nn.Module):
    def __init__(self, d=512, n=8, use_ckpt=False):
        super().__init__()
        self.blocks = nn.ModuleList([Block(d) for _ in range(n)])
        self.use_ckpt = use_ckpt
    def forward(self, x):
        for b in self.blocks:
            x = checkpoint(b, x, use_reentrant=False) if self.use_ckpt else b(x)
        return x


## Experiment


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(32, 512, device=device, requires_grad=True)

def measure(model):
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    y = model(x).sum()
    y.backward()
    peak = torch.cuda.max_memory_allocated() / 1e6 if device == 'cuda' else None
    return peak

for use_ckpt in [False, True]:
    model = Stack(d=512, n=8, use_ckpt=use_ckpt).to(device)
    peak = measure(model)
    print(f'use_ckpt={use_ckpt}  peak={peak} MB')


## Discussion

- Checkpointing a single module is the easiest start; for finer control segment the network into checkpoint groups.
- Reentrant checkpointing is the legacy API; use `use_reentrant=False` for the modern dispatcher.
- The savings stack with mixed precision and offloading (e.g. DeepSpeed ZeRO).


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
